In [2]:
import fitz  # PyMuPDF
import math
import csv
from collections import defaultdict

PDF_PATH = r"Karpinets_2006.pdf"
OUT_CSV = r"karpinets2006_fig2_vector_digitized.csv"

# --- helpers ---
def rect_center(r):
    return ((r.x0 + r.x1) / 2.0, (r.y0 + r.y1) / 2.0)

def seg_center(p1, p2):
    return ((p1.x + p2.x) / 2.0, (p1.y + p2.y) / 2.0)

def dist(a, b):
    return math.hypot(a[0]-b[0], a[1]-b[1])

def angle_deg(p1, p2):
    # angle of segment, normalized to [0,180)
    ang = math.degrees(math.atan2(p2.y - p1.y, p2.x - p1.x))
    ang = ang % 180.0
    return ang

def norm_angle(ang):
    # fold to [0,90]
    a = abs(ang)
    if a > 90:
        a = 180 - a
    return a

# Simple clustering (DBSCAN-like) without sklearn:
def cluster_points(points, eps=2.0):
    """
    points: list of dicts with keys: center=(x,y), payload=...
    returns list of clusters: each cluster is list of point dicts
    """
    clusters = []
    used = [False]*len(points)
    for i in range(len(points)):
        if used[i]:
            continue
        # grow cluster
        seed = points[i]
        used[i] = True
        cluster = [seed]
        changed = True
        while changed:
            changed = False
            for j in range(len(points)):
                if used[j]:
                    continue
                pj = points[j]
                # if close to ANY point already in cluster -> add
                if any(dist(pj["center"], pk["center"]) <= eps for pk in cluster):
                    used[j] = True
                    cluster.append(pj)
                    changed = True
        clusters.append(cluster)
    return clusters

# --- main ---
doc = fitz.open(PDF_PATH)

# Figure 2 is on page with caption "Figure 2" (in this PDF it's page index 4 = "Page 5 of 10")
# If your copy differs, change PAGE_IDX.
PAGE_IDX = 4
page = doc.load_page(PAGE_IDX)
drawings = page.get_drawings()

# A rough rectangle around Figure 2 plot area in page coordinates (points).
# This was derived from the PDF layout of this specific file.
# If needed, tweak slightly (x0,y0,x1,y1).
FIG2_PLOT_RECT = fitz.Rect(78.8, 499.6, 276.5, 615.7)

# 1) Find axis lines from thin strokes inside plot rect (width ~0.04)
thin = [d for d in drawings
        if fitz.Rect(d["rect"]).intersects(FIG2_PLOT_RECT)
        and d.get("type") == "s"
        and d.get("width") is not None
        and d["width"] < 0.1]

# Extract candidate long segments from thin strokes
h_lines = []
v_lines = []
for d in thin:
    for it in d["items"]:
        if it[0] != "l":
            continue
        _, p1, p2 = it
        # classify by orientation and length
        dx = abs(p2.x - p1.x)
        dy = abs(p2.y - p1.y)
        length = math.hypot(dx, dy)
        if length < 20:
            continue
        if dy < 1.0 and dx > 30:
            h_lines.append((p1, p2, length))
        if dx < 1.0 and dy > 30:
            v_lines.append((p1, p2, length))

# Choose the longest horizontal as x-axis, longest vertical as y-axis
if not h_lines or not v_lines:
    raise RuntimeError("Could not find axis lines automatically. Try adjusting FIG2_PLOT_RECT or PAGE_IDX.")

x_axis = max(h_lines, key=lambda t: t[2])
y_axis = max(v_lines, key=lambda t: t[2])

# axis coordinates
x0 = min(x_axis[0].x, x_axis[1].x)  # SGR=0
x1 = max(x_axis[0].x, x_axis[1].x)  # SGR=2
y_bottom = max(y_axis[0].y, y_axis[1].y)  # R=0
y_top = min(y_axis[0].y, y_axis[1].y)     # R=0.6

# 2) Collect marker primitives from thick strokes/fills (width ~0.324), excluding dashed regression lines
marker_draw = []
for d in drawings:
    r = fitz.Rect(d["rect"])
    if not r.intersects(FIG2_PLOT_RECT):
        continue
    # ignore dashed lines
    if d.get("dashes") and d["dashes"] != "[] 0":
        continue
    # keep fills or thick strokes
    if d.get("type") in ("f", "fs") or (d.get("type") == "s" and d.get("width", 0) >= 0.2):
        marker_draw.append(d)

# Convert drawing items into primitive "points" for clustering
prims = []
for d in marker_draw:
    drect = fitz.Rect(d["rect"])
    # ignore huge strokes (axis frame etc.) by size
    if drect.width > 220 or drect.height > 160:
        continue

    dtype = d["type"]
    fill = d.get("fill")
    color = d.get("color")
    width = d.get("width")

    for it in d["items"]:
        cmd = it[0]
        if cmd == "re":  # rectangle primitive
            _, rr, _ = it
            c = rect_center(rr)
            prims.append({
                "center": c,
                "kind": "rect",
                "rect": rr,
                "dtype": dtype,
                "fill": fill,
                "color": color,
                "width": width
            })
        elif cmd == "l":  # line segment
            _, p1, p2 = it
            c = seg_center(p1, p2)
            prims.append({
                "center": c,
                "kind": "seg",
                "p1": p1,
                "p2": p2,
                "dtype": dtype,
                "fill": fill,
                "color": color,
                "width": width
            })
        # ignore curves for this figure (usually not needed for these markers)

# 3) Cluster primitives into markers
clusters = cluster_points(prims, eps=2.2)

def map_to_data(cx, cy):
    # SGR axis: 0..2
    sgr = (cx - x0) / (x1 - x0) * 2.0
    # R axis: 0..0.6
    r = (y_bottom - cy) / (y_bottom - y_top) * 0.6
    return sgr, r

def classify_cluster(cluster):
    # Decide marker type and thus organism
    fills = [p for p in cluster if p.get("fill") is not None]
    rects = [p for p in cluster if p["kind"] == "rect"]
    segs  = [p for p in cluster if p["kind"] == "seg"]

    # cluster center (average)
    cx = sum(p["center"][0] for p in cluster) / len(cluster)
    cy = sum(p["center"][1] for p in cluster) / len(cluster)

    # Filled red triangle => N. crassa
    # In this PDF, red fill appears as (0.50, 0.0, 0.0) approx.
    if fills:
        # choose first fill color
        fc = fills[0].get("fill")
        if fc and fc[0] > 0.2 and fc[1] < 0.1 and fc[2] < 0.1:
            return "N. crassa", "filled triangle (red)", cx, cy

        # filled black rectangle => E. coli
        if rects and (fills[0].get("fill") == (0.0, 0.0, 0.0)):
            return "E. coli", "filled square", cx, cy

    # Open square rectangles (no fill, only stroke) => S. coelicolor
    if rects and not fills:
        return "S. coelicolor", "open square", cx, cy

    # Otherwise classify by segment orientations and closed-loop count
    # Compute orientations
    angs = []
    for s in segs:
        angs.append(norm_angle(angle_deg(s["p1"], s["p2"])))

    # If closed triangle (3-ish segments) => S. cerevisiae study 1
    if len(segs) in (3,4) and len(angs) >= 3:
        # triangle edges tend to have 3 distinct orientations
        if len({round(a/10)*10 for a in angs}) >= 3 and len(segs) <= 4:
            return "S. cerevisiae (study 1)", "open triangle", cx, cy

    # Open diamond (4 segments, ~45° rotated shape) => P. zopfii
    if len(segs) >= 4:
        # diamond edges often near ~45 and ~0 (after folding), but tends to have 4 segments and symmetric bbox
        # heuristic: if most angles are ~45±20
        if sum(1 for a in angs if abs(a-45) < 20) >= 3:
            return "P. zopfii", "open diamond", cx, cy

    # Plus: mostly 0 and 90 (folded => ~0 and ~90)
    if angs:
        has0 = any(a < 15 for a in angs)
        has90 = any(a > 75 for a in angs)
        has45 = any(abs(a-45) < 15 for a in angs)

        if has0 and has90 and not has45:
            return "S. ruminantium", "plus", cx, cy

        # X: diagonals only
        if has45 and not (has0 or has90):
            return "M. bovis", "x", cx, cy

        # Asterisk: mix of diagonal + orth
        if has45 and (has0 or has90):
            return "S. cerevisiae (study 2)", "asterisk", cx, cy

    return None, "unclassified", cx, cy

rows = []
for cl in clusters:
    species, note, cx, cy = classify_cluster(cl)
    if species is None:
        continue
    # keep only clusters whose centers lie well inside plot bounds
    if not (x0 - 1 <= cx <= x1 + 1 and y_top - 1 <= cy <= y_bottom + 1):
        continue
    sgr, r = map_to_data(cx, cy)
    # sanity bounds
    if not (0.0 <= sgr <= 2.0 and 0.0 <= r <= 0.6):
        continue
    rows.append({
        "growth_rate_hr": sgr,
        "mass_fraction": r,
        "source": species,
        "method": "vector_extracted_from_pdf",
        "notes": note
    })

# Deduplicate very-close points (some markers have both fill and stroke contributing)
rows_sorted = sorted(rows, key=lambda d: (d["source"], d["growth_rate_hr"], d["mass_fraction"]))
dedup = []
for d in rows_sorted:
    if not dedup:
        dedup.append(d); continue
    prev = dedup[-1]
    if (d["source"] == prev["source"]
        and abs(d["growth_rate_hr"] - prev["growth_rate_hr"]) < 0.005
        and abs(d["mass_fraction"] - prev["mass_fraction"]) < 0.005):
        continue
    dedup.append(d)

with open(OUT_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["growth_rate_hr","mass_fraction","source","method","notes"])
    w.writeheader()
    for d in dedup:
        w.writerow(d)

print(f"Wrote {len(dedup)} points to {OUT_CSV}")

Wrote 29 points to karpinets2006_fig2_vector_digitized.csv
